# Phase 05D: ExpansionNet v2 → S1 Sequence-KD Weight-Ablation Readiness

Notebook CPU này không huấn luyện. Nó khóa hai production contract seed 42:

- **R2:** `0.75 CE_human + 0.25 CE_teacher_sequence`;
- **R3:** `0.25 CE_human + 0.75 CE_teacher_sequence`.

Toàn bộ data/model/optimizer/scheduler/validation/checkpoint engine được kế thừa
nguyên vẹn từ Phase 04C đã pilot. Chỉ trọng số convex của hai loss đã kiểm chứng
được thay đổi. Mỗi full-run sau đó vẫn bắt buộc chạy discarded pilot tại chỗ
trước optimizer update đầu tiên.

## Inputs

1. Phase 05C Sequence-KD fallback entry.
2. Phase 04C ExpansionNet v2 → S1 training readiness.
3. Phase 04B ExpansionNet v2 → S1 preflight.

CPU, Internet Off; không gắn MS COCO/model/Test.


## Cell 1 — Authorization và immutable source discovery


In [1]:
import copy
import csv
import hashlib
import json
import os
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

STARTED=time.perf_counter()
INPUT_ROOT=Path(os.environ.get("KLCN_INPUT_ROOT","/kaggle/input"))
NOTEBOOK_ROOT=INPUT_ROOT/"notebooks" if (INPUT_ROOT/"notebooks").is_dir() else INPUT_ROOT
OUTPUT_ROOT=Path(os.environ.get("KLCN_OUTPUT_ROOT","/kaggle/working//phase5_teacher_a_s1_sequence_weight_readiness"))
AUDIT_DIR=OUTPUT_ROOT/"audits"; CONFIG_DIR=OUTPUT_ROOT/"configs"; MANIFEST_DIR=OUTPUT_ROOT/"manifests"
for d in [OUTPUT_ROOT,AUDIT_DIR,CONFIG_DIR,MANIFEST_DIR]: d.mkdir(parents=True,exist_ok=True)

def now_utc(): return datetime.now(timezone.utc).isoformat()
def read_json(p):
    with Path(p).open("r",encoding="utf-8") as h: return json.load(h)
def write_json(p,x):
    p=Path(p); t=p.with_suffix(p.suffix+".tmp")
    t.write_text(json.dumps(x,ensure_ascii=False,indent=2)+"\n",encoding="utf-8"); os.replace(t,p)
def sha256_file(p):
    d=hashlib.sha256()
    with Path(p).open("rb") as h:
        for b in iter(lambda:h.read(8*1024*1024),b""): d.update(b)
    return d.hexdigest()
def exactly_one(n):
    m=sorted(NOTEBOOK_ROOT.rglob(n)); assert len(m)==1,(n,m); return m[0].resolve()
def inside(root,rel):
    root=Path(root).resolve(); p=(root/rel).resolve(); assert p.is_relative_to(root); return p

P5C_PATH=exactly_one("PHASE5C_SEQUENCE_KD_FALLBACK_ENTRY_COMPLETE.json")
AUTH_PATH=exactly_one("PHASE5D_SEQUENCE_KD_WEIGHT_ABLATION_READINESS_AUTHORIZED.json")
P4C_PATH=exactly_one("PHASE4_TEACHER_A_S1_SEQUENCE_KD_TRAINING_READINESS_COMPLETE.json")
P4B_PATH=exactly_one("PHASE4_TEACHER_A_S1_SEQUENCE_KD_PREFLIGHT_COMPLETE.json")
P5C,AUTH,P4C,P4B=map(read_json,[P5C_PATH,AUTH_PATH,P4C_PATH,P4B_PATH])
assert P5C["status"]=="complete" and P5C["new_seed42_arms"]==["R2","R3"]
assert P5C["phase05d_authorized"] is True
assert AUTH["status"]=="authorized" and AUTH["allowed_seed"]==42
assert AUTH["full_training_authorized"] is False
assert P4C["status"]=="complete" and P4C["discarded_pilot"]=="passed"
assert P4C["production_contract"]=="locked"
assert P4B["status"]=="complete" and P4B["combined_loss_backward"]=="passed"
for x in [P5C,AUTH,P4C,P4B]:
    assert x["test_set_status"]=="sealed" and x["test_set_used"] is False

write_json(AUDIT_DIR/"phase5d_cell01_inputs.json",{
 "schema_version":"phase5d_cell01_v1","created_at_utc":now_utc(),"status":"passed",
 "phase5c_authorized":True,"phase4c_engine_ready":True,"phase4b_backward_ready":True,
 "gpu_required":False,"dataset_read":False,"full_training_started":False,
 "test_set_status":"sealed","test_set_used":False,"elapsed_seconds":time.perf_counter()-STARTED})
print("="*72); print("PHASE 05D CELL 1 STATUS"); print("="*72)
print("Status                    : PASSED")
print("Phase 05C / 04C / 04B     : VERIFIED")
print("GPU / dataset             : NOT REQUIRED / NOT READ")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 5 code cells")
print("Next                      : verify manifests and inherited engine")


PHASE 05D CELL 1 STATUS
Status                    : PASSED
Phase 05C / 04C / 04B     : VERIFIED
GPU / dataset             : NOT REQUIRED / NOT READ
Test Set status           : SEALED
Progress                  : 1 / 5 code cells
Next                      : verify manifests and inherited engine


## Cell 2 — Manifest integrity và production-engine inheritance


In [2]:
STARTED=time.perf_counter()
def verify(root,ref,n):
    p=inside(root,ref["relative_path"]); assert sha256_file(p)==ref["sha256"]
    with p.open("r",encoding="utf-8",newline="") as h: rows=list(csv.DictReader(h))
    assert len(rows)==n
    for r in rows:
        a=inside(root,r["relative_path"]); assert a.is_file()
        assert a.stat().st_size==int(r["size_bytes"]) and sha256_file(a)==r["sha256"]
    return rows
p5c_rows=verify(P5C_PATH.parent,P5C["manifest"]["csv"],P5C["manifest"]["csv"]["rows"])
p4c_rows=verify(P4C_PATH.parent,P4C["manifest"]["csv"],P4C["manifest_rows"])
p4b_rows=verify(P4B_PATH.parent,P4B["manifest"]["csv"],P4B["manifest"]["csv"]["rows"])
base_contract_path=inside(P4C_PATH.parent,"configs/phase4c_teacher_a_s1_sequence_kd_production_contract.json")
base=read_json(base_contract_path)
assert base["status"]=="locked" and base["kd"]["alpha"]==0.5
assert base["teacher"]["pseudo_rows"]==113287
assert base["training"]["physical_batch"]==32 and base["training"]["accumulation"]==2
assert base["validation"]["selection_primary"]=="CIDEr-D"

write_json(AUDIT_DIR/"phase5d_cell02_inheritance.json",{
 "schema_version":"phase5d_cell02_v1","created_at_utc":now_utc(),"status":"passed",
 "manifest_rows_verified":len(p5c_rows)+len(p4c_rows)+len(p4b_rows),
 "inherited_contract_sha256":sha256_file(base_contract_path),
 "inherited_components":["data pipeline","S1 initialization","AdamW","scheduler","FP16 GradScaler",
 "atomic checkpoint/resume","Beam-3 Validation","early stopping"],
 "test_set_status":"sealed","test_set_used":False,"elapsed_seconds":time.perf_counter()-STARTED})
print("="*72); print("PHASE 05D CELL 2 STATUS"); print("="*72)
print("Manifest rows verified    :",len(p5c_rows)+len(p4c_rows)+len(p4b_rows))
print("Production engine         : INHERITED EXACTLY")
print("Physical/effective batch  : 32 / 64")
print("Pseudo-caption rows       : 113287")
print("Validation primary        : CIDEr-D")
print("Progress                  : 2 / 5 code cells")
print("Next                      : audit convex loss deltas")


PHASE 05D CELL 2 STATUS
Manifest rows verified    : 25
Production engine         : INHERITED EXACTLY
Physical/effective batch  : 32 / 64
Pseudo-caption rows       : 113287
Validation primary        : CIDEr-D
Progress                  : 2 / 5 code cells
Next                      : audit convex loss deltas


## Cell 3 — R2/R3 loss arithmetic audit


In [3]:
STARTED=time.perf_counter()
loss_audit=read_json(inside(P4B_PATH.parent,"audits/phase4b_cell04_loss_backward.json"))
assert loss_audit["status"]=="passed" and loss_audit["student_gradients_finite"] is True
h=float(loss_audit["human_ce"]); t=float(loss_audit["teacher_sequence_ce"])
losses={"R1":0.5*h+0.5*t,"R2":0.75*h+0.25*t,"R3":0.25*h+0.75*t}
assert abs(losses["R1"]-float(loss_audit["combined_loss"]))<1e-6
assert all(min(h,t)<=x<=max(h,t) for x in losses.values())
assert all(x>0 for x in losses.values())
write_json(AUDIT_DIR/"phase5d_cell03_loss_weights.json",{
 "schema_version":"phase5d_cell03_v1","created_at_utc":now_utc(),"status":"passed",
 "source_human_ce":h,"source_teacher_sequence_ce":t,"weighted_losses":losses,
 "convex_bounds_passed":True,"constituent_backward_preflight":"passed",
 "run_local_discarded_pilot_required":True,"test_set_status":"sealed","test_set_used":False,
 "elapsed_seconds":time.perf_counter()-STARTED})
print("="*72); print("PHASE 05D CELL 3 — LOSS AUDIT"); print("="*72)
print("Human / teacher CE        :",h,"/",t)
print("R1 / R2 / R3 loss         :",losses["R1"],losses["R2"],losses["R3"])
print("Convex and finite         : PASSED")
print("Run-local pilot required  : True")
print("Progress                  : 3 / 5 code cells")
print("Next                      : lock R2 and R3 production contracts")


PHASE 05D CELL 3 — LOSS AUDIT
Human / teacher CE        : 10.975905418395996 / 11.158007621765137
R1 / R2 / R3 loss         : 11.066956520080566 11.021430969238281 11.112482070922852
Convex and finite         : PASSED
Run-local pilot required  : True
Progress                  : 3 / 5 code cells
Next                      : lock R2 and R3 production contracts


## Cell 4 — Lock separate R2/R3 seed-42 production contracts


In [4]:
STARTED=time.perf_counter()
specs={
 "R2":{"human":0.75,"teacher":0.25,"experiment_id":"P5A_EXPANSIONNET_S1_SEQKD_R2_W025"},
 "R3":{"human":0.25,"teacher":0.75,"experiment_id":"P5A_EXPANSIONNET_S1_SEQKD_R3_W075"},
}
contract_paths={}
for arm,s in specs.items():
    c=copy.deepcopy(base)
    c["contract_version"]="phase5d_sequence_weight_ablation_v1"
    c["created_at_utc"]=now_utc(); c["experiment_id"]=s["experiment_id"]
    c["phase"]="05D readiness"; c["arm"]=arm
    c["kd"]["alpha"]=s["teacher"]
    c["kd"]["loss"]=f'{s["human"]} CE_human + {s["teacher"]} CE_teacher_sequence'
    c["comparison"]={"primary_anchor":"R1 alpha=0.5 seed42","paired_seed":42,
      "cider_delta_minimum":0.001,"bleu4_non_regression_tolerance":-0.002}
    c["run_local_preflight"]={"discarded_optimizer_pilot_required":True,
      "initial_restore_exact_required":True,"pilot_weights_saved":False}
    c["immutable_from_phase4c"]=True; c["test_set_status"]="sealed"; c["test_set_used"]=False
    p=CONFIG_DIR/f"phase5d_{arm.lower()}_seed42_production_contract.json"
    write_json(p,c); contract_paths[arm]=p
assert sha256_file(contract_paths["R2"])!=sha256_file(contract_paths["R3"])
write_json(AUDIT_DIR/"phase5d_cell04_contracts.json",{
 "schema_version":"phase5d_cell04_v1","created_at_utc":now_utc(),"status":"passed",
 "contracts":{a:{"sha256":sha256_file(p),"relative_path":p.relative_to(OUTPUT_ROOT).as_posix()} for a,p in contract_paths.items()},
 "separate_output_roots_required":True,"seed":42,"full_training_started":False,
 "test_set_status":"sealed","test_set_used":False,"elapsed_seconds":time.perf_counter()-STARTED})
print("="*72); print("PHASE 05D CELL 4 STATUS"); print("="*72)
print("R2 contract               : LOCKED | human 0.75 / teacher 0.25")
print("R3 contract               : LOCKED | human 0.25 / teacher 0.75")
print("Initialization / seed     : SHARED / 42")
print("Output roots              : MUST BE SEPARATE")
print("Progress                  : 4 / 5 code cells")
print("Next                      : package and authorize R2/R3 full runs")


PHASE 05D CELL 4 STATUS
R2 contract               : LOCKED | human 0.75 / teacher 0.25
R3 contract               : LOCKED | human 0.25 / teacher 0.75
Initialization / seed     : SHARED / 42
Output roots              : MUST BE SEPARATE
Progress                  : 4 / 5 code cells
Next                      : package and authorize R2/R3 full runs


## Cell 5 — Manifest và dual full-run authorization


In [5]:
STARTED=time.perf_counter()
files=sorted(AUDIT_DIR.glob("*.json"))+sorted(CONFIG_DIR.glob("*.json"))
rows=[{"relative_path":p.relative_to(OUTPUT_ROOT).as_posix(),"size_bytes":p.stat().st_size,
       "sha256":sha256_file(p)} for p in files]
manifest=MANIFEST_DIR/"phase5d_sequence_weight_readiness_manifest.csv"
with manifest.open("w",encoding="utf-8",newline="") as h:
    w=csv.DictWriter(h,fieldnames=["relative_path","size_bytes","sha256"]); w.writeheader(); w.writerows(rows)
sums=MANIFEST_DIR/"SHA256SUMS"
sums.write_text("".join(f"{r['sha256']}  {r['relative_path']}\n" for r in rows),encoding="utf-8")
auths={}
for arm,next_phase in [("R2","Phase 05E"),("R3","Phase 05F")]:
    p=OUTPUT_ROOT/f"PHASE5_{'E' if arm=='R2' else 'F'}_{arm}_SEED42_FULL_RUN_AUTHORIZED.json"
    write_json(p,{"schema_version":"phase5_sequence_weight_seed42_authorized_v1",
      "created_at_utc":now_utc(),"status":"authorized","arm":arm,"seed":42,
      "experiment_id":specs[arm]["experiment_id"],"authorized_next":f"{next_phase} {arm} seed42 full run",
      "production_contract":{"relative_path":contract_paths[arm].relative_to(OUTPUT_ROOT).as_posix(),
       "sha256":sha256_file(contract_paths[arm])},"run_local_discarded_pilot_required":True,
      "full_training_started":False,"test_set_status":"sealed","test_set_used":False})
    auths[arm]=p
marker=OUTPUT_ROOT/"PHASE5D_SEQUENCE_KD_WEIGHT_ABLATION_READINESS_COMPLETE.json"
write_json(marker,{"schema_version":"phase5d_sequence_weight_readiness_complete_v1",
 "created_at_utc":now_utc(),"status":"complete","arms_ready":["R2","R3"],"seed":42,
 "production_engine_inherited":"Phase 04C exact","loss_weight_delta_audited":True,
 "r2_authorization_sha256":sha256_file(auths["R2"]),"r3_authorization_sha256":sha256_file(auths["R3"]),
 "manifest":{"csv":{"relative_path":manifest.relative_to(OUTPUT_ROOT).as_posix(),
 "sha256":sha256_file(manifest),"rows":len(rows)},"sha256sums":{"relative_path":sums.relative_to(OUTPUT_ROOT).as_posix(),
 "sha256":sha256_file(sums)}},"full_training_started":False,
 "test_set_status":"sealed","test_set_used":False})
print("="*72); print("PHASE 05D FINAL STATUS"); print("="*72)
print("Status                    : COMPLETE")
print("R2 / R3 seed42            : AUTHORIZED / AUTHORIZED")
print("Production engine         : PHASE 04C INHERITED EXACT")
print("Run-local pilot           : REQUIRED")
print("Full training started     : False")
print("Manifest rows             :",len(rows))
print("Test Set status           : SEALED")
print("Progress                  : 5 / 5 code cells")
print("Next                      : run Phase 05E R2 and Phase 05F R3 independently")


PHASE 05D FINAL STATUS
Status                    : COMPLETE
R2 / R3 seed42            : AUTHORIZED / AUTHORIZED
Production engine         : PHASE 04C INHERITED EXACT
Run-local pilot           : REQUIRED
Full training started     : False
Manifest rows             : 6
Test Set status           : SEALED
Progress                  : 5 / 5 code cells
Next                      : run Phase 05E R2 and Phase 05F R3 independently
